# 10 — Clases y OOP

Python es multiparadigma. OOP no es obligatorio pero es útil para modelar entidades con estado y comportamiento.

## Clase básica

In [1]:
class Producto:
    # Atributo de clase — compartido por todas las instancias
    iva = 0.21

    def __init__(self, nombre: str, precio: float, stock: int = 0):
        # Atributos de instancia — propios de cada objeto
        self.nombre = nombre
        self.precio = precio
        self.stock  = stock

    def precio_con_iva(self) -> float:
        return round(self.precio * (1 + self.iva), 2)

    def aplicar_descuento(self, pct: float) -> None:
        self.precio = round(self.precio * (1 - pct), 2)

    def __repr__(self) -> str:
        # __repr__ es la representación para developers (logs, REPL)
        return f'Producto({self.nombre!r}, {self.precio}, stock={self.stock})'

    def __str__(self) -> str:
        # __str__ es la representación para usuarios (print)
        return f'{self.nombre} — ${self.precio:,.2f} (stock: {self.stock})'


laptop = Producto('Laptop Pro 15', 899.99, stock=12)
print(laptop)               # usa __str__
print(repr(laptop))         # usa __repr__
print(laptop.precio_con_iva())

laptop.aplicar_descuento(0.10)
print(laptop)


Laptop Pro 15 — $899.99 (stock: 12)
Producto('Laptop Pro 15', 899.99, stock=12)
1088.99
Laptop Pro 15 — $809.99 (stock: 12)


## Properties

`@property` permite acceso a atributos calculados con sintaxis de campo (sin paréntesis). Útil para validación y cálculos derivados.

In [2]:
class Transaccion:
    def __init__(self, cantidad: int, precio_unitario: float):
        self._cantidad        = cantidad
        self._precio_unitario = precio_unitario

    @property
    def total(self) -> float:
        return round(self._cantidad * self._precio_unitario, 2)

    @property
    def cantidad(self) -> int:
        return self._cantidad

    @cantidad.setter
    def cantidad(self, valor: int) -> None:
        if valor < 0:
            raise ValueError('La cantidad no puede ser negativa')
        self._cantidad = valor


t = Transaccion(10, 89.99)
print(t.total)    # 899.9 — sin paréntesis aunque es un método
t.cantidad = 15
print(t.total)    # 1349.85
# t.cantidad = -1 # ValueError


899.9
1349.85


## Dunder methods

Métodos especiales que Python llama internamente. Permiten que las clases respondan a operadores y funciones built-in.

In [3]:
class Carrito:
    def __init__(self):
        self._items: list[tuple[str, float, int]] = []

    def agregar(self, nombre: str, precio: float, cantidad: int = 1) -> None:
        self._items.append((nombre, precio, cantidad))

    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def __contains__(self, nombre: str) -> bool:
        return any(item[0] == nombre for item in self._items)

    def __repr__(self) -> str:
        return f'Carrito({len(self)} items, total=${self.total:,.2f})'

    @property
    def total(self) -> float:
        return round(sum(p * c for _, p, c in self._items), 2)


carrito = Carrito()
carrito.agregar('Laptop', 899.99)
carrito.agregar('Mouse', 15.50, 2)

print(len(carrito))           # 2
print('Laptop' in carrito)    # True
print(carrito)                # usa __repr__

for nombre, precio, cant in carrito:
    print(f'  {nombre}: {cant} × ${precio}')


2
True
Carrito(2 items, total=$930.99)
  Laptop: 1 × $899.99
  Mouse: 2 × $15.5


## Herencia

In [4]:
class ProductoDigital(Producto):
    iva = 0.0   # atributo de clase sobreescrito — los digitales están exentos

    def __init__(self, nombre: str, precio: float, url_descarga: str):
        super().__init__(nombre, precio, stock=0)   # stock infinito para digitales
        self.url_descarga = url_descarga

    def __str__(self) -> str:
        return f'{super().__str__()} [digital]'


curso = ProductoDigital('Curso Python DS', 49.99, 'https://plataforma.io/curso-123')
print(curso)
print(curso.precio_con_iva())   # 49.99 — iva=0.0
print(isinstance(curso, Producto))          # True
print(isinstance(curso, ProductoDigital))   # True


Curso Python DS — $49.99 (stock: 0) [digital]
49.99
True
True


## dataclasses

Decorador que genera automáticamente `__init__`, `__repr__` y `__eq__` a partir de las anotaciones de tipo.

In [5]:
from dataclasses import dataclass, field

@dataclass
class Pedido:
    id_pedido:  str
    cliente:    str
    items:      list[str] = field(default_factory=list)   # mutable default seguro
    completado: bool = False

    @property
    def num_items(self) -> int:
        return len(self.items)


p1 = Pedido('ORD-001', 'María Torres', items=['Laptop', 'Mouse'])
p2 = Pedido('ORD-002', 'Carlos López')

print(p1)            # __repr__ generado automáticamente
print(p1.num_items)  # 2
print(p1 == p2)      # False — __eq__ generado automáticamente


Pedido(id_pedido='ORD-001', cliente='María Torres', items=['Laptop', 'Mouse'], completado=False)
2
False


---
## Resumen

| Concepto | Python |
|----------|--------|
| Constructor | `__init__(self, ...)` |
| Representación | `__repr__` (devs) y `__str__` (usuarios) |
| Atributo calculado | `@property` |
| Validación en setter | `@prop.setter` |
| Operadores | `__len__`, `__iter__`, `__contains__`, `__add__` |
| Herencia | `class B(A): super().__init__(...)` |
| Clase de datos | `@dataclass` |
